# Backpropagation & Computation Graphs — AI Lab Instructor Notebook

*Deep Learning · Medium*



4 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
$f

## Task 1: Dataset & Setup

# Backpropagation & Computation Graphs — Student Lab

Focus: implement a tiny scalar autodiff engine (micrograd-style) and use it to train an XOR MLP.

![Backpropagation through a tiny computation graph](/images/w6-d5-backprop-computation-graph.png)

![XOR dataset (class-colored)](/images/w6-d4-xor-dataset.svg)

## Section 0 — XOR dataset

We’ll train a tiny MLP to solve XOR. If gradients are wrong, training won’t improve.

In [ ]:
import math
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

def make_xor(n=256, noise=0.15, seed=0):
    r = np.random.default_rng(seed)
    X = r.uniform(-1, 1, size=(n, 2))
    y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(int)
    X = X + r.normal(0, noise, size=X.shape)
    return X.astype(float), y

X, y = make_xor()
check('X_shape', X.shape == (256, 2))
check('y_shape', y.shape == (256,))
print('base_rate', float(y.mean()))

**Why this works.** # Backpropagation & Computation Graphs — Instructor Lab

Solutions + short rationales. Focus: reverse-mode autodiff (chain rule on a DAG) and training XOR with a tiny MLP.

## Section 0 — XOR dataset

Rationale: XOR is the minimal dataset that requires nonlinearity, so it’s a strong end-to-end test of gradients + optimization.

## Task 2: Implement `finite_diff(f, x, eps)` returning df/dx for scalar f(x).
*Section: Gradient checks (finite differences)*

## Section 1 — Gradient checks (finite differences)

We’ll numerically approximate gradients to validate `.backward()`.

### Multiply grad rule intuition

![Multiply grad rule intuition](/images/w6-d5-multiply-grad-rule.png)

### Task 1.1
Implement `finite_diff(f, x, eps)` returning df/dx for scalar f(x).

### Task 1.2
Check gradients for a composite function.

In [ ]:
def finite_diff(f, x, eps=1e-6):
    return (f(x + eps) - f(x - eps)) / (2 * eps)

def composite(x):
    xv = Value(x)
    out = (xv * xv + 2.0 * xv).tanh()
    out.backward()
    return out.data, xv.grad

x0 = 0.37
f0, g_ad = composite(x0)
g_fd = finite_diff(lambda t: composite(t)[0], x0)
print('grad_autodiff', g_ad, 'grad_fd', g_fd)
check('grad_close', abs(g_ad - g_fd) < 5e-4)

In [ ]:
# Checks
check('grad_close', abs(g_ad - g_fd) < 5e-4)

**Why this works.** ## Section 1 — Gradient checks

Rationale: finite-difference gradient checks catch sign mistakes, missing gradient accumulation, and wrong local derivatives.

## Task 3: Implement `Neuron.__call__(x)` computing an affine transform + activation.
*Section: Build an MLP from primitives*

## Section 2 — Build an MLP from primitives

We’ll build `Neuron`, `Layer`, `MLP` using lists of `Value`.

### Why `Value` (micrograd-style) matters

Micrograd gives an abstraction layer: in forward pass it composes `X·W + b` at hidden/output layers (with nonlinear activation like ReLU/sigmoid), and in backward pass it applies chain rule node-by-node to compute gradients and update weights.

![Chain rule for one neuron](/images/w6-d5-chain-rule-neuron.png)

- It makes autodiff explicit: each scalar stores both value and gradient.
- You can debug gradients node-by-node instead of treating backprop as a black box.
- Once `+`, `*`, and `relu` are correct, larger models are just graph composition.
- This is the same core idea modern frameworks use internally, just at tensor scale.

### Task 2.1
Implement `Neuron.__call__(x)` computing an affine transform + activation.

### Task 2.2
Implement `parameters()` for all modules.

In [ ]:
$11

In [ ]:
# Checks
check('param_nonempty', len(params) > 0)

**Why this works.** ## Section 2 — MLP from primitives

Rationale: once you have scalar autodiff, higher-level NN modules become thin wrappers around graph construction.

### Why `Value` matters

Rationale: this abstraction converts backprop from a formula-memorization exercise into a reusable engine. Build it once, and every composed model gets gradients automatically via the graph.

Micrograd gives an abstraction layer: in forward pass it composes `X·W + b` at hidden/output layers (with nonlinear activation like ReLU/sigmoid), and in backward pass it applies chain rule node-by-node to compute gradients and update weights.

## Task 4: Implement `softplus(v)` using `Value` ops.
*Section: Train XOR*

## Section 3 — Train XOR

We’ll use a stable-ish logistic loss on a single logit.

We represent labels as {-1, +1}. Loss per example: `softplus(-y * logit)` where `softplus(z)=log(1+exp(z))`.

### Task 3.1
Implement `softplus(v)` using `Value` ops.

HINT: you can implement `exp` approximately via `math.exp` on scalar values for this lab, by defining an `exp()` method. We'll include it for you below.

### Task 3.2
Use the MLP to train XOR and verify that loss decreases and accuracy becomes reasonable.

**Hints**
- you can implement `exp` approximately via `math.exp` on scalar values for this lab, by defining an `exp()` method. We'll include it for you below.

In [ ]:
$13

In [ ]:
# Checks
check('softplus_grad_close', abs(g_ad - g_fd) < 5e-4)
check('loss_decreased', loss1 < loss0)
check('acc_reasonable', accuracy() > 0.70)

**Why this works.** ## Section 3 — Train XOR

Rationale: training is a stress test that composes everything: forward graph build, backward pass, and parameter updates.